# RNA配列と構造をbpseqファイルに変換

## 準備

In [1]:
# ライブラリインポート
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json

# ワーキングディレクトリの変更
os.chdir(
    "/workspaces/MLMvsData2vec/"
)  # 変更したいディレクトリのパスを指定
# 現在の作業ディレクトリを確認
print("Current working directory:", os.getcwd())

# 出力ディレクトリの設定
output_dir = Path("results/notebook/convert_bpseq")
output_dir.mkdir(parents=True, exist_ok=True)

Current working directory: /workspaces/MLMvsData2vec


### ファミリーの指定など

In [2]:
# ファミリーの指定 ("5s", "16s", "23s", "grp1", "RNaseP", "srp", "telomerase", "tmRNA", "tRNA")
FAMILY = ["5s", "16s", "23s", "grp1", "RNaseP", "srp", "telomerase", "tmRNA", "tRNA"] # リストに登録したファミリーのみ対象とする. 空リストの場合は全てのファミリーを対象とする.

# score_diffの閾値
SCORE_DIFF_THRESHOLD = 0.5

# score_avgの閾値
SCORE_AVG_THRESHOLD = 0.1

FRAMEWORKS = ["data2vec_mlm", "data2vec", "mlm"]

### パスの設定

In [11]:
expt_name = "ArchiveII_famfold"  # 実験名 ArchiveII_famfold, ArchiveII_kfold
score_name = "f1"  # スコア名 f1, precision, recall

results_dir = Path("./results/SS_results/ArchiveII_famfold/")

test_timestamps = {
    "data2vec_mlm": "20260402T000234",
    "data2vec": "20260403T071832",
    "mlm": "20260404T123256",
}
result_paths = {
    "data2vec_mlm": results_dir / "data2vec/20260307T153057/knotfold/20260402T000234/overall_results.csv",
    "data2vec": results_dir / "data2vec/20260324T045257/knotfold/20260403T071832/overall_results.csv",
    "mlm": results_dir / "mlm/20260316T030756/knotfold/20260404T123256/overall_results.csv",
}

### 変換関数

In [4]:
def convert_to_bpseq(sequence, base_pairs):
    """
    塩基対のリストをBPSEQ形式に変換する関数
    Args:
        sequence (str): RNAの塩基配列
        base_pairs (list(list(str))): 塩基対のリスト
    Returns:
        str: BPSEQ形式の文字列
    """
    length = len(sequence)
    bpseq = []
    pair_dict = {i: j for i, j in base_pairs}
    pair_dict.update({j: i for i, j in base_pairs})

    for i in range(1, length + 1):
        pair = pair_dict.get(i, 0)
        bpseq.append(f"{i} {sequence[i - 1]} {pair}")

    return "\n".join(bpseq)

### ファイルの読み込み

In [12]:
# ファイルの読み込み
result_dfs = {
    "data2vec_mlm": pd.read_csv(result_paths["data2vec_mlm"]),
    "data2vec": pd.read_csv(result_paths["data2vec"]),
    "mlm": pd.read_csv(result_paths["mlm"]),
}

# 正解構造ファイル
ArchiveII_df = pd.read_csv("./data/SS_data/ArchiveII.csv", header=0)

## BPP行列を可視化している配列の変換

### RNAの指定

In [15]:
seq_ids = {
    fam: None for fam in FAMILY
}

# フレームワークは1つだけみる
for framework in FRAMEWORKS:
    # BPP行列を可視化している配列の特定
    for fam in FAMILY:
        fam_dir = result_paths[framework].parent / fam / "test_results" / test_timestamps[framework]
        # 存在確認
        if not fam_dir.exists():
            print(f"Directory {fam_dir} does not exist.")
            continue
        
        # fam_dir以下のpngファイルを探索し、ファイル名がfamで始まるものを1つ探す
        for file in fam_dir.glob("*.png"):
            if file.stem.startswith(fam):
                seq_ids[fam] = file.stem.removesuffix("_probability_matrix")
                break
        
    break

print(seq_ids)

{'5s': '5s_Acholeplasma-laidlawii-1', '16s': '16s_C.psittaci_domain4', '23s': '23s_H.pylori_domain6', 'grp1': 'grp1_a.I1.e.P.inouyei.C1.SSU.1506', 'RNaseP': 'RNaseP_A.brierleyi', 'srp': 'srp_Vibr.fisc._CP000020', 'telomerase': 'telomerase_AF221906.96-545', 'tmRNA': 'tmRNA_Stre.gord._TRW-29390_1-349', 'tRNA': 'tRNA_tdbR00000055-Schizosaccharomyces_pombe-4896-Glu-3UC'}


### bpseqへの変換

In [18]:
# frameworkの設定
frameworks = ["gt"]
frameworks.extend(FRAMEWORKS)

for framework in frameworks:
    # 予測結果ファイルの読み込み
    if framework == "gt":
        preds_df = ArchiveII_df
    else:
        preds_df = result_dfs[framework]

    for fam, seq_id in seq_ids.items():
        seq = preds_df.loc[preds_df["id"] == seq_id, "sequence"].values[0]
        base_pairs = json.loads(
            preds_df.loc[preds_df["id"] == seq_id, "base_pairs"].values[0]
        )
        bpseq_str = convert_to_bpseq(seq, base_pairs)

        # BPSEQファイルとして保存
        output_file_path = output_dir / f"{fam}" / f"{framework}_{seq_id}.bpseq"
        output_file_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_file_path, "w") as f:
            f.write(bpseq_str)
        print(f"BPSEQ file saved for {framework} at {output_file_path}")
        if framework != "gt":
            print(f"{score_name} score for {framework} on {seq_id}: {preds_df.loc[preds_df['id'] == seq_id, score_name].values[0]}")

BPSEQ file saved for gt at results/notebook/convert_bpseq/5s/gt_5s_Acholeplasma-laidlawii-1.bpseq
BPSEQ file saved for gt at results/notebook/convert_bpseq/16s/gt_16s_C.psittaci_domain4.bpseq
BPSEQ file saved for gt at results/notebook/convert_bpseq/23s/gt_23s_H.pylori_domain6.bpseq
BPSEQ file saved for gt at results/notebook/convert_bpseq/grp1/gt_grp1_a.I1.e.P.inouyei.C1.SSU.1506.bpseq
BPSEQ file saved for gt at results/notebook/convert_bpseq/RNaseP/gt_RNaseP_A.brierleyi.bpseq
BPSEQ file saved for gt at results/notebook/convert_bpseq/srp/gt_srp_Vibr.fisc._CP000020.bpseq
BPSEQ file saved for gt at results/notebook/convert_bpseq/telomerase/gt_telomerase_AF221906.96-545.bpseq
BPSEQ file saved for gt at results/notebook/convert_bpseq/tmRNA/gt_tmRNA_Stre.gord._TRW-29390_1-349.bpseq
BPSEQ file saved for gt at results/notebook/convert_bpseq/tRNA/gt_tRNA_tdbR00000055-Schizosaccharomyces_pombe-4896-Glu-3UC.bpseq
BPSEQ file saved for data2vec_mlm at results/notebook/convert_bpseq/5s/data2vec_ml

## 指定した配列の変換

In [ ]:
# RNAの指定
seq_id = "tRNA_tdbR00000522-Bos_taurus-9913-Ini-AU"

# frameworkの設定
frameworks = ["gt"]
frameworks.extend(FRAMEWORKS)
fam = seq_id.split("_")[0]

for framework in frameworks:
    # 予測結果ファイルの読み込み
    if framework == "gt":
        preds_df = ArchiveII_df
    else:
        preds_df = result_dfs[framework]

    seq = preds_df.loc[preds_df["id"] == seq_id, "sequence"].values[0]
    base_pairs = json.loads(
        preds_df.loc[preds_df["id"] == seq_id, "base_pairs"].values[0]
    )
    bpseq_str = convert_to_bpseq(seq, base_pairs)

    # BPSEQファイルとして保存
    output_file_path = output_dir / "secfied" / f"{framework}_{seq_id}.bpseq"
    output_file_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file_path, "w") as f:
        f.write(bpseq_str)
    print(f"BPSEQ file saved for {framework} at {output_file_path}")
    print(f"{score_name} score for {framework} on {seq_id}: {preds_df.loc[preds_df['id'] == seq_id, score_name].values[0]}")